# 13. API Testing Notebook

Tests internal APIs including the **v2 benchmark suite** and evaluation metrics.
Stack: AWS Bedrock (see `config/config.json`).

In [ ]:
import sys, os
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

test_results = {"passed": 0, "failed": 0, "details": []}

def run_test(name, func):
    try:
        if func() in [None, True]:
            test_results["passed"] += 1
            print(f"✅ {name}")
            return True
    except Exception as e:
        test_results["details"].append({"name": name, "error": str(e)})
    test_results["failed"] += 1
    print(f"❌ {name}")
    return False

In [ ]:
print("=" * 50)
print("📋 CONFIG & IMPORT TESTS")
print("=" * 50)

run_test("Config Import", lambda: __import__('src.cirq_rag_code_assistant.config', fromlist=['get_config']))
run_test("RAG Imports", lambda: all([__import__('src.rag.embeddings'), __import__('src.rag.retriever')]))
run_test("Agent Imports", lambda: all([__import__('src.agents.designer'), __import__('src.agents.validator')]))
run_test("Evaluation Imports", lambda: __import__('src.evaluation.metrics'))
run_test("CLI Imports", lambda: __import__('src.cli.main'))

In [ ]:
print("\n" + "=" * 50)
print("🔧 COMPONENT TESTS")
print("=" * 50)

from src.rag.embeddings import EmbeddingModel
from src.rag.vector_store import VectorStore
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever
from src.rag.generator import Generator

run_test("EmbeddingModel", lambda: EmbeddingModel().get_embedding_dimension() > 0)

em = EmbeddingModel()
run_test("VectorStore", lambda: VectorStore(em.get_embedding_dimension()) is not None)

vs = VectorStore(em.get_embedding_dimension())
kb = KnowledgeBase(embedding_model=em, vector_store=vs)
run_test("KnowledgeBase", lambda: kb is not None)

# Load KB
try:
    kb.load_from_directory()
    kb.load_index()
    print(f"   Loaded {len(kb.entries)} entries")
except: pass

run_test("Retriever", lambda: Retriever(kb) is not None)
run_test("Generator", lambda: Generator(Retriever(kb)) is not None)

In [ ]:
print("\n" + "=" * 50)
print("🤖 AGENT TESTS")
print("=" * 50)

from src.agents.designer import DesignerAgent
from src.agents.optimizer import OptimizerAgent  
from src.agents.validator import ValidatorAgent
from src.agents.educational import EducationalAgent

retriever = Retriever(kb)
generator = Generator(retriever)

run_test("DesignerAgent", lambda: DesignerAgent(retriever, generator) is not None)
run_test("OptimizerAgent", lambda: OptimizerAgent(retriever=retriever) is not None)
run_test("ValidatorAgent", lambda: ValidatorAgent(retriever=retriever) is not None)
run_test("EducationalAgent", lambda: EducationalAgent(retriever) is not None)

In [ ]:
print("\n" + "=" * 50)
print("📊 EVALUATION & BENCHMARK TESTS")
print("=" * 50)

from src.evaluation.benchmark import load_benchmark_prompts, STANDARD_BENCHMARKS
from src.evaluation.metrics import compute_code_quality_score, compute_statistics, wilson_ci

run_test("load_benchmark_prompts", lambda: len(load_benchmark_prompts()) == 25)
run_test("exclude_explanation tier", lambda: len(load_benchmark_prompts(exclude_explanation=True)) == 20)
run_test("STANDARD_BENCHMARKS fallback", lambda: len(STANDARD_BENCHMARKS) >= 4)

sample_code = "import cirq\nq = cirq.LineQubit.range(2)\nc = cirq.Circuit(cirq.H(q[0]), cirq.measure(q[0], key='m'))"
validation = {"compilation": {"success": True, "circuit": None}, "validation_passed": True}
score = compute_code_quality_score(sample_code, validation)
run_test("compute_code_quality_score", lambda: 0 <= score["code_quality_score"] <= 1)

stats = compute_statistics([0.8, 0.9, 0.85, 0.88, 0.92])
run_test("compute_statistics", lambda: stats["n"] == 5 and stats["mean"] > 0)

ci = wilson_ci(8, 10)
run_test("wilson_ci", lambda: 0 <= ci["ci_lower"] <= ci["mean"] <= ci["ci_upper"] <= 1)

from src.evaluation.ablation import AblationStudy, VARIANT_LABELS
run_test("AblationStudy import", lambda: len(VARIANT_LABELS) == 6)


In [ ]:
print("\n" + "=" * 50)
print("🎼 ORCHESTRATOR & VALIDATION TESTS")
print("=" * 50)

from src.orchestration.orchestrator import Orchestrator

designer = DesignerAgent(retriever, generator)
optimizer = OptimizerAgent(retriever=retriever)
validator = ValidatorAgent(retriever=retriever)
educational = EducationalAgent(retriever)

run_test("Orchestrator Init", lambda: Orchestrator(
    designer=designer, optimizer=optimizer, 
    validator=validator, educational=educational) is not None)

# Test validator with valid code
valid_code = "import cirq\nq = cirq.LineQubit.range(2)\nc = cirq.Circuit(cirq.H(q[0]))"
run_test("Validator (valid code)", lambda: validator.run({"code": valid_code}).get('validation_passed', False))

# Test validator with invalid code  
run_test("Validator (invalid code)", lambda: not validator.run({"code": "@@invalid@@"}).get('validation_passed', True))

In [ ]:
print("\n" + "=" * 50)
print("📊 TEST SUMMARY")
print("=" * 50)

total = test_results["passed"] + test_results["failed"]
rate = test_results["passed"] / total if total > 0 else 0

print(f"\nTotal: {total} | Passed: {test_results['passed']} | Failed: {test_results['failed']}")
print(f"Pass Rate: {rate:.1%}")

if test_results["details"]:
    print("\nFailed tests:")
    for d in test_results["details"]:
        print(f"  - {d['name']}: {d.get('error', 'Unknown')}")